# AI Enrichment and Tools

Wintermute integrates with multiple LLM providers (AWS Bedrock, OpenAI, Groq, HuggingFace)
through a unified **Router** system. AI capabilities include:
- Simple chat conversations
- Tool-calling (function calling) with automatic execution loops
- Hardware processor enrichment
- Custom tool registration

> **Note**: Most examples in this notebook require valid API credentials (AWS, OpenAI, or Groq).
> Code is shown for reference with comments indicating where credentials are needed.

## Router Initialization

The **Router** selects which LLM provider handles each request. `init_router()` registers
all available providers and returns a configured Router. Providers are stored in the
global `llms` registry.

In [ ]:
# Initialize the AI router (requires AWS credentials for Bedrock, or API keys for others)
# Environment variables: AWS_REGION, GROQ_API_KEY, OPENAI_API_KEY, BEDROCK_MODEL_ID
#
# router = init_router()
# print(f"Available providers: {llms.providers()}")
# print(f"Default provider: {router.default_provider}")

print("Router setup shown for reference — requires API credentials")
print("  init_router() -> Router")
print("  Registers: Bedrock, Groq, OpenAI, HuggingFace")
print("  Also bootstraps RAG knowledge bases")

## simple_chat — Basic LLM Conversations

`simple_chat()` sends a single prompt to the LLM and returns the response content as a string.
It routes through the Router to select the appropriate provider.

In [ ]:
# simple_chat takes a router and a prompt string:
#
# response = simple_chat(router, "What is JTAG and why is it a security concern?")
# print(response)

print("simple_chat(router, prompt, task_tag='generic') -> str")
print("  - router: Router instance from init_router()")
print("  - prompt: plain text question or instruction")
print("  - task_tag: routing hint ('cheap' routes to Groq)")
print("  - Returns: response content as a string")

## Tool Registration

Wintermute's **tool system** lets you register Python functions as LLM-callable tools.
`register_tools()` introspects function signatures and type hints to generate JSON Schema
definitions automatically.

Tools are stored in the global `tools` registry and made available to `tool_calling_chat()`.

In [ ]:
from wintermute.ai.tools_runtime import tools
from wintermute.ai.utils.tool_factory import register_tools


def lookup_cve(cve_id: str) -> str:
    """Look up a CVE by its ID and return vulnerability details."""
    return f"CVE {cve_id}: Example vulnerability description"


def check_port(host: str, port: int) -> str:
    """Check if a network port is open on the given host."""
    return f"Port {port} on {host}: open (simulated)"


registered = register_tools([lookup_cve, check_port])

# Register in the global registry so tool_calling_chat can find them
for tool in registered:
    tools.register(tool)

print(f"Registered tools: {[t.name for t in registered]}")
print(f"Tool definitions: {len(tools.get_definitions())} total")

# Test calling a tool directly
result = tools.call("lookup_cve", {"cve_id": "CVE-2024-1234"})
print(f"Direct call result: {result}")

## Processor Enrichment

`enrich_processor()` uses AI to fill in missing hardware details. Given a processor with
just a name and manufacturer, it queries the LLM for architecture, core count, security
features, and known vulnerabilities.

In [ ]:
from wintermute.hardware import Processor

proc = Processor(processor="STM32F407", manufacturer="STMicroelectronics")
print(f"Before enrichment: {proc.processor}")
print(f"  Architecture: {proc.architecture}")
print(f"  Endianness: {proc.endianness}")

# Enrichment uses AI to fill in missing hardware details:
# enriched = enrich_processor(proc, router=router)
# print(f"After: {enriched.architecture.instruction_set}, {enriched.architecture.cpu_cores} cores")

print("\nenrich_processor(processor, router=None) -> Processor")
print("  AI analyzes the processor name/manufacturer and fills:")
print("  - architecture (core, instruction_set, cpu_cores, key_features)")
print("  - endianness, processor_family")
print("  - overall_capabilities, pinout")

## tool_calling_chat — LLM with Function Calling

`tool_calling_chat()` runs the full tool-calling loop:
1. Send messages + available tool specs to the LLM
2. LLM decides which tools to call (if any)
3. Execute tool calls, feed results back as tool messages
4. Repeat until the LLM gives a final answer

This powers autonomous AI workflows like processor enrichment and vulnerability analysis.

In [ ]:
from wintermute.ai.tools_runtime import spec_from_tool

# Convert registered Tools to ToolSpecs for the LLM
tool_specs = [spec_from_tool(t) for t in registered]
print(f"Tool specs for LLM: {[s.name for s in tool_specs]}")

# Full tool-calling chat (requires active router):
#
# messages = [Message(role="user", content="Look up CVE-2024-1234 and check port 443 on 10.0.1.10")]
# response = tool_calling_chat(router, messages, tools=tool_specs)
# print(response.content)

print("\ntool_calling_chat(router, messages, tools, ...) -> ChatResponse")
print("  Automatically executes tool calls in a loop")
print("  Returns ChatResponse with .content and .tool_calls")

## Summary

Wintermute's AI subsystem provides:
- **Multi-provider routing** — Bedrock, OpenAI, Groq, HuggingFace behind a single API
- **Simple chat** — one-shot prompt/response via `simple_chat()`
- **Tool calling** — automatic function execution loops via `tool_calling_chat()`
- **Hardware enrichment** — AI-powered processor analysis via `enrich_processor()`
- **Custom tools** — register any Python function as an LLM-callable tool

See `04-AI-Enrichment-and-Tools` for RAG knowledge bases that augment AI with domain documents.